# 21 Channeled / interdigitated / spiral bifilar / pleated (EP4602674A1)

Channel $3$–$50\,\mathrm{mm}$ × $1$–$40\,\mathrm{mm}$, spacing $10$–$50\,\mathrm{mm}$, Fe $1$–$7\,\mathrm{g\,cm^{-2}}$, VF $0.5$–$0.9$.

## Governing equations

See the requirement UID and patent claims cited in the title cell. Default electrolyte $6\,\mathrm{M}$ KOH at $303\,\mathrm{K}$ and $1\,\mathrm{atm}$. Currents in $\mathrm{A}$, voltages in $\mathrm{V}$, power in $\mathrm{W}$. Chemistry remains $\mathrm{...}$ (not mhchem).


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

here = Path.cwd()
for cand in (here, here.parent):
    if (cand / "src" / "ironair").is_dir() and (cand / "plant_sim").is_dir():
        sys.path.insert(0, str(cand))
        sys.path.insert(0, str(cand / "src"))
        break

from plant_sim.bootstrap import setup_path

setup_path()
from plant_sim.params import CAPACITY_FE_OH2_MAH_G, CAPACITY_MAGNETITE_MAH_G, default_params

In [ ]:
from IPython import get_ipython

_ipy = get_ipython()
if _ipy is not None:
    _ipy.run_line_magic("matplotlib", "inline")

sns.set(
    rc={
        "axes.labelsize": 12,
        "axes.titlesize": 18,
        "figure.figsize": (11, 8.5),
        "figure.dpi": 300,
        "figure.facecolor": "w",
        "figure.edgecolor": "k",
    }
)
P = default_params()
print("KOH default", P.c_KOH_mol_m3 / 1000, "M; T_ep", P.T_ep_sim_K, "K")
print("Faraday 960/320 checks", round(CAPACITY_FE_OH2_MAH_G, 2), round(CAPACITY_MAGNETITE_MAH_G, 2))

## Simulation

In [ ]:
from plant_sim.substructures.channeled_electrode import ChanneledElectrode

ch = ChanneledElectrode(P)
cmp = ch.compare_layouts(10 * P.I_discharge_100h_A)
print({k: round(v, 4) for k, v in cmp.items() if k.startswith("CE_")})
ce_frame = pd.DataFrame(
    {
        "layout": [k[3:] for k in cmp if k.startswith("CE_")],
        "CE": [v for k, v in cmp.items() if k.startswith("CE_")],
    }
)
fig, ax = plt.subplots()
sns.barplot(data=ce_frame, x="layout", y="CE", ax=ax)
ax.set_xlabel("layout")
ax.set_ylabel("CE")
ax.set_title("Charge CE vs electrode layout")
ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
res = ch.simulate((0, 400), u={"I_fe_A": -10 * P.I_discharge_100h_A})
print("finite", np.all(np.isfinite(res["y"])))